In [1]:
from analysis import load_metrics
from datetime import datetime
import pandas as pd
import sqlite3 as sql
from contextlib import closing
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
DB_PATH = os.environ["DB_PATH"]

## Aggregating Meditation (value type is none)


In [4]:
with closing(sql.connect(DB_PATH)) as conn:
    meditation = load_metrics(conn, 2, "Meditate", "US/Pacific")
meditation

,recorded_at
62,2026-02-08 17:05:48-08:00
63,2026-02-10 14:16:26-08:00
64,2026-02-12 11:12:37-08:00
65,2026-02-13 09:17:49-08:00
66,2026-02-20 12:52:02-08:00
67,2026-02-21 13:49:30-08:00
68,2026-02-23 06:20:21-08:00
69,2026-02-24 07:26:40-08:00
70,2026-03-02 16:44:26-08:00
71,2026-03-03 10:21:11-08:00


Lets make sure that the datetimes returned are actually converted to a new timezone, i.e., the timestamp is kept as-is, but the timezones are changed, instead of localized or attached, where the timestamp is changed to keep the datetime as-is.


In [5]:
with closing(sql.connect(DB_PATH)) as conn:
    tp = load_metrics(conn, 2, "Meditate", "Asia/Kolkata")
tp

,recorded_at
62,2026-02-09 06:35:48+05:30
63,2026-02-11 03:46:26+05:30
64,2026-02-13 00:42:37+05:30
65,2026-02-13 22:47:49+05:30
66,2026-02-21 02:22:02+05:30
67,2026-02-22 03:19:30+05:30
68,2026-02-23 19:50:21+05:30
69,2026-02-24 20:56:40+05:30
70,2026-03-03 06:14:26+05:30
71,2026-03-03 23:51:11+05:30


In [7]:
with closing(sql.connect(DB_PATH)) as conn:
    tp = load_metrics(conn, 2, "Meditate", "UTC")
tp

,recorded_at
62,2026-02-09 01:05:48+00:00
63,2026-02-10 22:16:26+00:00
64,2026-02-12 19:12:37+00:00
65,2026-02-13 17:17:49+00:00
66,2026-02-20 20:52:02+00:00
67,2026-02-21 21:49:30+00:00
68,2026-02-23 14:20:21+00:00
69,2026-02-24 15:26:40+00:00
70,2026-03-03 00:44:26+00:00
71,2026-03-03 18:21:11+00:00


In [8]:
meditation.info()

<class 'pandas.DataFrame'>
Index: 12 entries, 62 to 73
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype                    
---  ------       --------------  -----                    
 0   recorded_at  12 non-null     datetime64[s, US/Pacific]
dtypes: datetime64[s, US/Pacific](1)
memory usage: 192.0 bytes


### Aggregating by Timeseries

Grouping by timestamps is done via the `.resample()` method. It downsamples by-second data into daily/weekly/monthly/etc. aggregates. It is useful to keep the split-apply-combine paradigm in mind. The splitting is done by timestamps, once the data is split into groups, I need to apply some function to the group. By default the function is applied to all the columns in the group. I can specify that the function should only be applied to numeric columns, etc., but the usual pattern I have seen is that you usually first select the columns you want to apply the function to, then split the projected dataframe and apply the function.


In [9]:
# Split: On the weekly recorded_at field
# Apply: The size function which will count the number of columns in the dataframes. Even though
# apart from recorded_on, there are no other columns, it is ok. The count function will not work
# because it needs other columns to count the non-missing values in them.
# The result is a series because there no other columns.
medgrps = meditation.resample("W", on="recorded_at").size()
print(type(medgrps))
medgrps

<class 'pandas.Series'>


recorded_at
2026-02-08 00:00:00-08:00    1
2026-02-15 00:00:00-08:00    3
2026-02-22 00:00:00-08:00    2
2026-03-01 00:00:00-08:00    2
2026-03-08 00:00:00-08:00    4
Freq: W-SUN, dtype: int64

In [10]:
# Here is what count will give me
meditation.resample("W", on="recorded_at").count()

""
recorded_at
2026-02-08 00:00:00-08:00
2026-02-15 00:00:00-08:00
2026-02-22 00:00:00-08:00
2026-03-01 00:00:00-08:00
2026-03-08 00:00:00-08:00


In [11]:
# Iterating through the series will give 2-tuple, the index value and the data value
for idx, sz in medgrps.items():
    print(idx, sz)

2026-02-08 00:00:00-08:00 1
2026-02-15 00:00:00-08:00 3
2026-02-22 00:00:00-08:00 2
2026-03-01 00:00:00-08:00 2
2026-03-08 00:00:00-08:00 4


In [12]:
# I can just get the index
medgrps.index

DatetimeIndex(['2026-02-08 00:00:00-08:00', '2026-02-15 00:00:00-08:00',
               '2026-02-22 00:00:00-08:00', '2026-03-01 00:00:00-08:00',
               '2026-03-08 00:00:00-08:00'],
              dtype='datetime64[s, US/Pacific]', name='recorded_at', freq='W-SUN')

## Aggregating Weights (numeric value)


In [13]:
with closing(sql.connect(DB_PATH)) as conn:
    weights = load_metrics(conn, 2, "Weight", "US/Pacific")
weights

,recorded_at,value
74,2026-02-06 19:02:36-08:00,139.0
75,2026-02-08 10:56:37-08:00,151.0
76,2026-02-09 17:42:44-08:00,132.0
77,2026-02-11 07:30:00-08:00,159.0
78,2026-02-13 07:54:40-08:00,130.0
79,2026-02-15 10:35:11-08:00,191.0
80,2026-02-17 16:30:16-08:00,183.0
81,2026-02-22 18:41:37-08:00,120.0
82,2026-02-27 19:44:26-08:00,174.0
83,2026-03-02 14:43:26-08:00,117.0


In [14]:
# Unlike meditation groups, this one is a dataframe because there was at least one additional column
# apart from recorded_at
wtgrps = weights.resample("W", on="recorded_at").mean()
print(type(wtgrps))
wtgrps

<class 'pandas.DataFrame'>


,value
recorded_at,
2026-02-08 00:00:00-08:00,145.0
2026-02-15 00:00:00-08:00,153.0
2026-02-22 00:00:00-08:00,151.5
2026-03-01 00:00:00-08:00,174.0
2026-03-08 00:00:00-08:00,164.0


In [15]:
# When I iterate through any dataframe using items(), it will yeild each column
# as a series. The column will consist of a 2-tuple, the name of the column and
# the series. In this case there is only a single column named value.
for colname, series in wtgrps.items():
    print(f"*** {colname} ****")

    # Now I can iterate through the series as usual
    for idx, avg in series.items():
        print(idx, avg)

*** value ****
2026-02-08 00:00:00-08:00 145.0
2026-02-15 00:00:00-08:00 153.0
2026-02-22 00:00:00-08:00 151.5
2026-03-01 00:00:00-08:00 174.0
2026-03-08 00:00:00-08:00 164.0


In [16]:
# There is a second way I can iterate through the dataframe and that is using
# iterrows, though the API is a bit funky. The iterator will yeild a 2-tuple, the
# first element is the index value, and the second element is a series. The series
# will contain each column name as the index, and the column value as the row value.
# So this series will have as many rows as the number of columns in the original df.
# In this case there is only a single column, so the series will contain a single row
# with the index as the colname and the row value as the value of that column.
for idx, val in wtgrps.iterrows():
    print(idx)
    for validx, valval in val.items():
        print(validx, valval)
    print("---")

2026-02-08 00:00:00-08:00
value 145.0
---
2026-02-15 00:00:00-08:00
value 153.0
---
2026-02-22 00:00:00-08:00
value 151.5
---
2026-03-01 00:00:00-08:00
value 174.0
---
2026-03-08 00:00:00-08:00
value 164.0
---


## Aggregating Mood (categorical value)


In [18]:
with closing(sql.connect(DB_PATH)) as conn:
    mood = load_metrics(conn, 2, "Mood", "US/Pacific")
mood.head(15)

,recorded_at,value
1,2026-02-06 07:56:22-08:00,Angry
2,2026-02-06 18:05:07-08:00,Angry
4,2026-02-07 06:32:34-08:00,Angry
3,2026-02-07 12:15:53-08:00,Serene
5,2026-02-08 18:07:46-08:00,Sad
6,2026-02-09 10:32:33-08:00,Angry
7,2026-02-09 11:55:58-08:00,Angry
8,2026-02-09 13:44:59-08:00,Angry
11,2026-02-11 09:23:31-08:00,Angry
10,2026-02-11 10:07:25-08:00,Angry


In [19]:
# Again, count will not work because I am consuming both the columns - recorded_at and value - as
# group keys. After that there are no more columns left to count.
moodgrps = mood.groupby([pd.Grouper(key="recorded_at", freq="W"), "value"]).size()
print(type(moodgrps))
moodgrps

<class 'pandas.Series'>


recorded_at                value 
2026-02-08 00:00:00-08:00  Sad       1
                           Angry     3
                           Serene    1
2026-02-15 00:00:00-08:00  Happy     3
                           Sad       1
                           Angry     7
                           Serene    4
2026-02-22 00:00:00-08:00  Happy     6
                           Sad       2
                           Angry     1
                           Serene    3
2026-03-01 00:00:00-08:00  Happy     7
                           Sad       6
                           Angry     3
                           Serene    1
2026-03-08 00:00:00-08:00  Happy     1
                           Sad       2
                           Angry     3
                           Serene    6
dtype: int64

In [20]:
# I can "pivot" the resulting series, but I don't expect to use this.
mood.groupby([pd.Grouper(key="recorded_at", freq="W"), "value"]).size().unstack(
    fill_value=0
)

value,Happy,Sad,Angry,Serene
recorded_at,,,,
2026-02-08 00:00:00-08:00,0,1,3,1
2026-02-15 00:00:00-08:00,3,1,7,4
2026-02-22 00:00:00-08:00,6,2,1,3
2026-03-01 00:00:00-08:00,7,6,3,1
2026-03-08 00:00:00-08:00,1,2,3,6


In [21]:
# Here the index is a Multi-Index, i.e., it is composed of multiple elements
# in this case two, the timeseries key, and the value key.
for index, sz in moodgrps.items():
    print(index, sz)

(Timestamp('2026-02-08 00:00:00-0800', tz='US/Pacific'), 'Sad') 1
(Timestamp('2026-02-08 00:00:00-0800', tz='US/Pacific'), 'Angry') 3
(Timestamp('2026-02-08 00:00:00-0800', tz='US/Pacific'), 'Serene') 1
(Timestamp('2026-02-15 00:00:00-0800', tz='US/Pacific'), 'Happy') 3
(Timestamp('2026-02-15 00:00:00-0800', tz='US/Pacific'), 'Sad') 1
(Timestamp('2026-02-15 00:00:00-0800', tz='US/Pacific'), 'Angry') 7
(Timestamp('2026-02-15 00:00:00-0800', tz='US/Pacific'), 'Serene') 4
(Timestamp('2026-02-22 00:00:00-0800', tz='US/Pacific'), 'Happy') 6
(Timestamp('2026-02-22 00:00:00-0800', tz='US/Pacific'), 'Sad') 2
(Timestamp('2026-02-22 00:00:00-0800', tz='US/Pacific'), 'Angry') 1
(Timestamp('2026-02-22 00:00:00-0800', tz='US/Pacific'), 'Serene') 3
(Timestamp('2026-03-01 00:00:00-0800', tz='US/Pacific'), 'Happy') 7
(Timestamp('2026-03-01 00:00:00-0800', tz='US/Pacific'), 'Sad') 6
(Timestamp('2026-03-01 00:00:00-0800', tz='US/Pacific'), 'Angry') 3
(Timestamp('2026-03-01 00:00:00-0800', tz='US/Pacific

In [23]:
# I can index with the first index to get all the elements that match it.
tp = moodgrps[pd.Timestamp('2026-02-08 00:00:00-0800', tz='US/Pacific')]
print(type(tp))
print(tp)

<class 'pandas.Series'>
value
Sad       1
Angry     3
Serene    1
dtype: int64


In [24]:
# Just get the first index
moodgrps.index.get_level_values(0).unique()

DatetimeIndex(['2026-02-08 00:00:00-08:00', '2026-02-15 00:00:00-08:00',
               '2026-02-22 00:00:00-08:00', '2026-03-01 00:00:00-08:00',
               '2026-03-08 00:00:00-08:00'],
              dtype='datetime64[s, US/Pacific]', name='recorded_at', freq=None)

In [25]:
# This does not seem very useful
moodgrps.index.get_level_values(1)

CategoricalIndex(['Sad', 'Angry', 'Serene', 'Happy', 'Sad', 'Angry', 'Serene',
                  'Happy', 'Sad', 'Angry', 'Serene', 'Happy', 'Sad', 'Angry',
                  'Serene', 'Happy', 'Sad', 'Angry', 'Serene'],
                 categories=['Happy', 'Sad', 'Angry', 'Serene'], ordered=False, dtype='category', name='value')

In [26]:
for week in moodgrps.index.get_level_values(0).unique():
    print(f"In the week ending {week}:")
    for moodcat, count in moodgrps[week].items():
        print(f"\tI was {moodcat} {count} number of times.")

In the week ending 2026-02-08 00:00:00-08:00:
	I was Sad 1 number of times.
	I was Angry 3 number of times.
	I was Serene 1 number of times.
In the week ending 2026-02-15 00:00:00-08:00:
	I was Happy 3 number of times.
	I was Sad 1 number of times.
	I was Angry 7 number of times.
	I was Serene 4 number of times.
In the week ending 2026-02-22 00:00:00-08:00:
	I was Happy 6 number of times.
	I was Sad 2 number of times.
	I was Angry 1 number of times.
	I was Serene 3 number of times.
In the week ending 2026-03-01 00:00:00-08:00:
	I was Happy 7 number of times.
	I was Sad 6 number of times.
	I was Angry 3 number of times.
	I was Serene 1 number of times.
In the week ending 2026-03-08 00:00:00-08:00:
	I was Happy 1 number of times.
	I was Sad 2 number of times.
	I was Angry 3 number of times.
	I was Serene 6 number of times.


## Aggregating Meal (categorical properties but no value)


In [ ]:
meal = load_metrics(1, "Meal", "US/Pacific")
meal.head(20)

In [ ]:
mealgrps = meal.groupby([pd.Grouper(key="recorded_at", freq="W"), "healthy"]).size()
mealgrps

In [ ]:
# How healthy were home cooked delicious meals?
meal[(meal["source"] == "home-cooked") & (meal["taste"] == "delicious")].groupby(
    [pd.Grouper(key="recorded_at", freq="W"), "healthy"]
).size()

## Aggregating Blood Glucose (categorical properties and numeric value)


In [ ]:
glucose = load_metrics(1, "Blood Glucose", "US/Pacific")
glucose.head(10)

In [ ]:
glucose[["recorded_at", "value", "delta"]].groupby(
    [pd.Grouper(key="recorded_at", freq="W"), "delta"]
).mean()

In [ ]:
# How are my sugar levels around breakfast?
glucose.loc[glucose["event"] == "breakfast", ["recorded_at", "value", "delta"]].groupby(
    [pd.Grouper(key="recorded_at", freq="W"), "delta"]
).mean()

## Aggregate Hike (mixed properties, numeric value)


In [ ]:
hike = load_metrics(1, "Hike", "US/Pacific")
hike

In [ ]:
hike["elevation_gain_bin"] = pd.cut(
    hike["elevation_gain"], bins=3, labels=["low", "medium", "high"]
)
hike

In [ ]:
hike[["recorded_at", "value", "elevation_gain_bin"]].groupby(
    [pd.Grouper(key="recorded_at", freq="W"), "elevation_gain_bin"]
).mean()

# 